# morphological_quantification_2026-01-02 — 01_per_z_whole_morph_candidates

**Feeds:** Fig 3f

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# 01 | Per-Z Whole-Morph Candidate Outlines

Generate a whole-morph outline independently on every z plane for every image. This notebook is intentionally upstream of any decision about whether the final morph representation should come from one z plane, a merged mask, or a projection.


## Cell Guide

- `Setup`: resolve the project root, import helper code, and define output paths.
- `Load Manifest`: read the unified image manifest for this cohort.
- `Representative Otsu Parameter Sweep`: compare a few DAPI-only Otsu variants on representative image/z pairs before the full batch run.
- `Per-Z Candidate Generation`: segment the morph independently on each z plane and save per-plane mask candidates.
- `Per-Z Review Pages`: render image-first review pages with brightfield and DAPI plus mask outlines across z.
- `Likely Outlier Review`: surface the likeliest mask outliers and show their fit on brightfield and DAPI.
- `Next Step`: move to `02` for per-z geometry review.


In [ ]:
import math
import os
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tifffile
from IPython.display import Image, Markdown, display

if Path.cwd().name == "notebooks":
    ROOT = Path.cwd().resolve().parent
elif (Path.cwd() / "notebooks").exists():
    ROOT = Path.cwd().resolve()
else:
    raise RuntimeError("Run this notebook from the project root or the notebooks/ directory.")

SCRIPTS_DIR = ROOT / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import morphology_quantification_helpers as mqh


## Settings Notes

- This notebook does **not** lock a chosen z plane.
- The whole-morph outline is computed from the DAPI plane only, independently on every z plane rather than on a projection.
- Thresholding is intentionally Otsu-based, with baseline subtraction. The current default candidate uses a slightly relaxed Otsu scale plus a small dilation because plain Otsu was too tight on first pass.
- The primary review artifact here is image output: one review page per file with whole-morph outlines and threshold intermediates across z.
- `OPTIMIZATION_MODE` can be turned on later to suppress heavy inline rendering while keeping the saved outputs on disk.


In [ ]:
ANALYSIS_MANIFEST_PATH = ROOT / "results" / "manifests" / "analysis_manifest.tsv"
CANDIDATE_TABLE_PATH = ROOT / "results" / "tables" / "whole_morph_z_candidates.tsv"
MASK_DIR = ROOT / "results" / "masks" / "whole_morph_z_candidates"
QC_DIR = ROOT / "results" / "qc" / "whole_morph_z_candidates"
DEBUG_QC_DIR = ROOT / "results" / "qc" / "whole_morph_segmentation_debug"
SWEEP_QC_DIR = ROOT / "results" / "qc" / "whole_morph_otsu_parameter_sweep"
OUTLIER_QC_DIR = ROOT / "results" / "qc" / "whole_morph_outlier_review"

OPTIMIZATION_MODE = False
RENDER_INLINE_REVIEW_PAGES = not OPTIMIZATION_MODE
RENDER_INLINE_DEBUG_EXAMPLES = not OPTIMIZATION_MODE
DEBUG_EXAMPLE_FILE_IDS = [1, 12, 20]
MAX_INLINE_REVIEW_PAGES = 3
OUTLIER_FILE_COUNT = 4
PARAMETER_SWEEP_CASES = [
    {"file_id": 1, "z_index": 6},
    {"file_id": 2, "z_index": 0},
    {"file_id": 7, "z_index": 4},
]
PARAMETER_SWEEP_VARIANTS = [
    {"label": "otsu x1.00 | dil 0", "threshold_scale": 1.00, "dilation_radius_px": 0},
    {"label": "otsu x0.90 | dil 0", "threshold_scale": 0.90, "dilation_radius_px": 0},
    {"label": "otsu x0.80 | dil 2", "threshold_scale": 0.80, "dilation_radius_px": 2},
    {"label": "otsu x0.70 | dil 3", "threshold_scale": 0.70, "dilation_radius_px": 3},
    {"label": "otsu x0.60 | dil 4", "threshold_scale": 0.60, "dilation_radius_px": 4},
    {"label": "otsu x0.50 | dil 5", "threshold_scale": 0.50, "dilation_radius_px": 5},
    {"label": "otsu x0.40 | dil 6", "threshold_scale": 0.40, "dilation_radius_px": 6},
    {"label": "otsu x0.30 | dil 7", "threshold_scale": 0.30, "dilation_radius_px": 7},
]
WHOLE_MORPH_PARAMS = {
    "gaussian_sigma": 2.0,
    "baseline_percentile": 5.0,
    "threshold_scale": 0.50,
    "min_size_px": 4000,
    "opening_radius_px": 3,
    "closing_radius_px": 7,
    "dilation_radius_px": 5,
}

MASK_DIR.mkdir(parents=True, exist_ok=True)
QC_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_QC_DIR.mkdir(parents=True, exist_ok=True)
SWEEP_QC_DIR.mkdir(parents=True, exist_ok=True)
OUTLIER_QC_DIR.mkdir(parents=True, exist_ok=True)


## Load Manifest


In [ ]:
manifest_df = pd.read_csv(ANALYSIS_MANIFEST_PATH, sep="\t")
if "include_in_analysis" in manifest_df.columns:
    manifest_df = manifest_df[manifest_df["include_in_analysis"].fillna(True).astype(bool)].copy()
manifest_df = manifest_df.sort_values(["file_id", "file_path"]).reset_index(drop=True)
records = mqh.records_from_manifest(manifest_df)

display(manifest_df[[
    "file_id",
    "acquisition_date",
    "condition_label",
    "acquisition_batch_label",
]].head(12))
print(f"Images queued for per-z outlining: {len(records)}")


## Representative Otsu Parameter Sweep


In [ ]:
sweep_paths = []
for case in PARAMETER_SWEEP_CASES:
    file_id = int(case["file_id"])
    z_idx = int(case["z_index"])
    hit = manifest_df.loc[manifest_df["file_id"] == file_id].copy()
    if hit.empty:
        print(f"Skipping missing file_id={file_id}")
        continue

    row = hit.iloc[0]
    abs_path = ROOT / str(row.file_path)
    plane = mqh.load_plane_channels(abs_path, z_index=z_idx)
    bf = plane["brightfield"]
    dapi = plane["dapi"]
    if dapi is None:
        raise RuntimeError(f"Missing DAPI channel for {row.file_path}")

    bf_display = (
        mqh.robust_rescale(bf)
        if bf is not None
        else np.zeros_like(dapi, dtype=np.float32)
    )
    dapi_display = mqh.robust_rescale(dapi)

    fig, axes = plt.subplots(
        3,
        len(PARAMETER_SWEEP_VARIANTS),
        figsize=(2.7 * len(PARAMETER_SWEEP_VARIANTS), 11.2),
        constrained_layout=True,
    )
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[:, None]

    for col_idx, variant in enumerate(PARAMETER_SWEEP_VARIANTS):
        params = dict(WHOLE_MORPH_PARAMS)
        params["threshold_scale"] = float(variant["threshold_scale"])
        params["dilation_radius_px"] = int(variant["dilation_radius_px"])
        mask, seg_debug = mqh.segment_whole_morph_with_debug(dapi, **params)
        area_fraction = float(mask.mean())
        threshold_value = float(seg_debug["threshold_info"]["threshold_value"])
        excluded_dapi_display = np.array(dapi_display, copy=True)
        excluded_dapi_display[mask] = 0.0

        row_payloads = [
            (bf_display, "BF", True),
            (dapi_display, "DAPI", True),
            (excluded_dapi_display, "Excluded DAPI", False),
        ]

        for row_idx, (display_img, label, draw_outline) in enumerate(row_payloads):
            ax = axes[row_idx, col_idx]
            ax.imshow(display_img, cmap="gray", vmin=0.0, vmax=1.0)
            if draw_outline:
                mqh.plot_mask_outline(ax, mask, color="yellow", linewidth=1.25)
            if row_idx == 0:
                ax.set_title(variant["label"], fontsize=10)
            if col_idx == 0:
                ax.set_ylabel(label, fontsize=10)
            if row_idx < 2:
                ax.text(
                    0.5,
                    -0.10,
                    f"thr={threshold_value:.0f} | area={area_fraction:.3f}",
                    transform=ax.transAxes,
                    ha="center",
                    va="top",
                    fontsize=8,
                )
            ax.set_xticks([])
            ax.set_yticks([])

    fig.suptitle(
        f"Representative Otsu sweep | file {file_id:02d} | z={z_idx} | {row.acquisition_date}",
        fontsize=12,
    )
    out_path = SWEEP_QC_DIR / f"{file_id:02d}_z{z_idx:02d}_otsu_parameter_sweep.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    sweep_paths.append(out_path)

print(f"Wrote {len(sweep_paths)} parameter-sweep figures to {SWEEP_QC_DIR}")
if RENDER_INLINE_DEBUG_EXAMPLES:
    for path in sweep_paths:
        print(path.relative_to(ROOT))
        display(Image(filename=str(path)))
else:
    print("Inline parameter-sweep display skipped because OPTIMIZATION_MODE=True.")


## Per-Z Candidate Generation


In [ ]:
candidate_rows = []
page_paths = []

for record in records:
    abs_path = ROOT / record.file_path
    stack = mqh.load_czi_stack(abs_path)
    z_count = int(stack.data_czyx.shape[1])
    pixel_size_um = stack.scale_um.get("X")

    fig, axes = plt.subplots(
        5,
        z_count,
        figsize=(max(2.4 * z_count, 8.0), 11.0),
        constrained_layout=True,
    )
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[:, None]

    for z_idx in range(z_count):
        plane = mqh.load_plane_channels(abs_path, z_index=z_idx)
        dapi = plane["dapi"]
        bf = plane["brightfield"]
        if dapi is None:
            raise RuntimeError(f"Missing DAPI channel for {record.file_path}")

        mask, seg_debug = mqh.segment_whole_morph_with_debug(dapi, **WHOLE_MORPH_PARAMS)
        mask_path = MASK_DIR / f"{int(record.file_id):02d}_z{z_idx:02d}_whole_morph_mask.tif"
        tifffile.imwrite(mask_path, mask.astype(np.uint8))

        has_mask = bool(mask.any())
        if has_mask:
            geometry = mqh.measure_mask_geometry(mask, pixel_size_um=pixel_size_um)
        else:
            geometry = {
                "area_px": 0,
                "area_fraction": 0.0,
                "major_axis_length_px": np.nan,
                "major_axis_length_um": np.nan,
                "minor_axis_length_px": np.nan,
                "minor_axis_length_um": np.nan,
                "aspect_ratio": np.nan,
                "eccentricity": np.nan,
                "solidity": np.nan,
                "touches_border": False,
            }

        candidate_rows.append(
            {
                "image_id": record.image_id,
                "cohort_id": record.cohort_id,
                "canonical_position": record.canonical_position,
                "file_id": int(record.file_id),
                "file_path": record.file_path,
                "acquisition_date": record.acquisition_date,
                "acquisition_batch_label": record.acquisition_batch_label,
                "z_index": int(z_idx),
                "z_count": int(z_count),
                "mask_path": str(mask_path.relative_to(ROOT)),
                "has_mask": has_mask,
                "baseline_value": float(seg_debug["baseline_value"]),
                "threshold_method": str(seg_debug["threshold_info"]["method"]),
                "threshold_scale": float(seg_debug["threshold_info"]["threshold_scale"]),
                "threshold_value": float(seg_debug["threshold_info"]["threshold_value"]),
                "raw_component_count": int(seg_debug["n_raw_components"]),
                "area_px": int(geometry["area_px"]),
                "area_fraction": float(geometry["area_fraction"]),
                "major_axis_length_um": float(geometry["major_axis_length_um"]),
                "minor_axis_length_um": float(geometry["minor_axis_length_um"]),
                "aspect_ratio": float(geometry["aspect_ratio"]),
                "eccentricity": float(geometry["eccentricity"]),
                "solidity": float(geometry["solidity"]),
                "touches_border": bool(geometry["touches_border"]),
            }
        )

        bf_display = (
            mqh.robust_rescale(bf)
            if bf is not None
            else np.zeros_like(dapi, dtype=np.float32)
        )
        dapi_display = mqh.robust_rescale(dapi)
        flat_display = mqh.robust_rescale(seg_debug["flat"])
        threshold_mask = np.asarray(seg_debug["mask_threshold"], dtype=np.float32)
        mask_after_morphology = np.asarray(seg_debug["mask_after_morphology"], dtype=np.float32)

        ax_bf = axes[0, z_idx]
        ax_bf.imshow(bf_display, cmap="gray")
        mqh.plot_mask_outline(ax_bf, mask, color="yellow", linewidth=1.0)
        ax_bf.set_title(f"z={z_idx}", fontsize=8)
        ax_bf.set_xticks([])
        ax_bf.set_yticks([])
        if z_idx == 0:
            ax_bf.set_ylabel("BF", fontsize=9)

        ax_dapi = axes[1, z_idx]
        ax_dapi.imshow(dapi_display, cmap="gray")
        mqh.plot_mask_outline(ax_dapi, mask, color="yellow", linewidth=1.0)
        ax_dapi.set_xticks([])
        ax_dapi.set_yticks([])
        if z_idx == 0:
            ax_dapi.set_ylabel("DAPI", fontsize=9)

        ax_flat = axes[2, z_idx]
        ax_flat.imshow(flat_display, cmap="gray")
        ax_flat.set_xticks([])
        ax_flat.set_yticks([])
        if z_idx == 0:
            ax_flat.set_ylabel("Flat DAPI", fontsize=9)

        ax_thr = axes[3, z_idx]
        ax_thr.imshow(threshold_mask, cmap="gray")
        ax_thr.set_xticks([])
        ax_thr.set_yticks([])
        if z_idx == 0:
            ax_thr.set_ylabel("Otsu mask", fontsize=9)

        ax_morph = axes[4, z_idx]
        ax_morph.imshow(mask_after_morphology, cmap="gray")
        mqh.plot_mask_outline(ax_morph, mask, color="yellow", linewidth=1.0)
        subtitle = (
            f"thr={float(seg_debug['threshold_info']['threshold_value']):.1f}\n"
            f"area={geometry['area_fraction']:.3f}"
            if has_mask
            else "empty mask"
        )
        ax_morph.text(
            0.5,
            -0.10,
            subtitle,
            transform=ax_morph.transAxes,
            ha="center",
            va="top",
            fontsize=7,
        )
        ax_morph.set_xticks([])
        ax_morph.set_yticks([])
        if z_idx == 0:
            ax_morph.set_ylabel("Morph + final", fontsize=9)

    fig.suptitle(
        f"{record.canonical_position} | file {int(record.file_id):02d} | {record.acquisition_date}",
        fontsize=12,
    )
    page_path = QC_DIR / f"{int(record.file_id):02d}_all_z_whole_morph_candidates.png"
    fig.savefig(page_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    page_paths.append(page_path)

candidate_df = pd.DataFrame(candidate_rows).sort_values(["file_id", "z_index"]).reset_index(drop=True)
candidate_df.to_csv(CANDIDATE_TABLE_PATH, sep="\t", index=False)

display(candidate_df[[
    "file_id",
    "z_index",
    "threshold_scale",
    "threshold_value",
    "area_fraction",
    "aspect_ratio",
    "touches_border",
]].head(18))
print(f"Wrote {len(candidate_df)} per-z candidate rows to {CANDIDATE_TABLE_PATH}")
print(f"Wrote {len(page_paths)} review pages to {QC_DIR}")


## Per-Z Review Pages


In [ ]:
if RENDER_INLINE_REVIEW_PAGES:
    preview_ids = sorted(candidate_df["file_id"].unique())[:MAX_INLINE_REVIEW_PAGES]
    for file_id in preview_ids:
        page_path = QC_DIR / f"{int(file_id):02d}_all_z_whole_morph_candidates.png"
        print(page_path.relative_to(ROOT))
        display(Image(filename=str(page_path)))
else:
    print("Inline review page display skipped because OPTIMIZATION_MODE=True.")


## Representative Segmentation Debug


In [ ]:
debug_page_paths = []
for file_id in DEBUG_EXAMPLE_FILE_IDS:
    sub = candidate_df.loc[candidate_df["file_id"] == int(file_id)].copy()
    if sub.empty:
        continue
    row = sub.sort_values(["area_fraction", "z_index"], ascending=[False, True]).iloc[0]
    abs_path = ROOT / str(row.file_path)
    z_idx = int(row.z_index)
    plane = mqh.load_plane_channels(abs_path, z_index=z_idx)
    mask, seg_debug = mqh.segment_whole_morph_with_debug(plane["dapi"], **WHOLE_MORPH_PARAMS)
    title = (
        f"file {int(file_id):02d} | z={z_idx} | "
        f"{row.acquisition_date} | Otsu={float(seg_debug['threshold_info']['threshold_value']):.1f}"
    )
    fig = mqh.plot_whole_morph_segmentation_debug_figure(
        brightfield=plane["brightfield"],
        dapi=plane["dapi"],
        mask=mask,
        debug=seg_debug,
        title=title,
    )
    debug_path = DEBUG_QC_DIR / f"{int(file_id):02d}_representative_segmentation_debug.png"
    fig.savefig(debug_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    debug_page_paths.append(debug_path)

print(f"Wrote {len(debug_page_paths)} representative debug figures to {DEBUG_QC_DIR}")
if RENDER_INLINE_DEBUG_EXAMPLES:
    for debug_path in debug_page_paths:
        print(debug_path.relative_to(ROOT))
        display(Image(filename=str(debug_path)))
else:
    print("Inline debug-example display skipped because OPTIMIZATION_MODE=True.")


## Likely Outlier Review


In [ ]:
def robust_abs_score(series: pd.Series) -> pd.Series:
    values = series.astype(float).to_numpy()
    finite_mask = np.isfinite(values)
    scores = np.zeros_like(values, dtype=float)
    if finite_mask.sum() < 3:
        return pd.Series(scores, index=series.index)
    finite_values = values[finite_mask]
    median = float(np.median(finite_values))
    mad = float(np.median(np.abs(finite_values - median)))
    if mad < 1e-8:
        return pd.Series(scores, index=series.index)
    scores[finite_mask] = np.abs(finite_values - median) / (1.4826 * mad)
    return pd.Series(scores, index=series.index)


def robust_high_score(series: pd.Series) -> pd.Series:
    values = series.astype(float).to_numpy()
    finite_mask = np.isfinite(values)
    scores = np.zeros_like(values, dtype=float)
    if finite_mask.sum() < 3:
        return pd.Series(scores, index=series.index)
    finite_values = values[finite_mask]
    median = float(np.median(finite_values))
    mad = float(np.median(np.abs(finite_values - median)))
    if mad < 1e-8:
        return pd.Series(scores, index=series.index)
    scores[finite_mask] = np.clip((finite_values - median) / (1.4826 * mad), 0.0, None)
    return pd.Series(scores, index=series.index)


def robust_low_score(series: pd.Series) -> pd.Series:
    values = series.astype(float).to_numpy()
    finite_mask = np.isfinite(values)
    scores = np.zeros_like(values, dtype=float)
    if finite_mask.sum() < 3:
        return pd.Series(scores, index=series.index)
    finite_values = values[finite_mask]
    median = float(np.median(finite_values))
    mad = float(np.median(np.abs(finite_values - median)))
    if mad < 1e-8:
        return pd.Series(scores, index=series.index)
    scores[finite_mask] = np.clip((median - finite_values) / (1.4826 * mad), 0.0, None)
    return pd.Series(scores, index=series.index)


best_candidate_df = (
    candidate_df
    .sort_values(
        ["file_id", "area_fraction", "solidity", "z_index"],
        ascending=[True, False, False, True],
    )
    .groupby("file_id", as_index=False)
    .head(1)
    .copy()
)

best_candidate_df["area_outlier_score"] = robust_abs_score(best_candidate_df["area_fraction"])
best_candidate_df["solidity_low_score"] = robust_low_score(best_candidate_df["solidity"])
best_candidate_df["components_high_score"] = robust_high_score(best_candidate_df["raw_component_count"])
best_candidate_df["border_score"] = best_candidate_df["touches_border"].astype(float) * 2.5
best_candidate_df["edge_z_score"] = (
    (best_candidate_df["z_index"] <= 0)
    | (best_candidate_df["z_index"] >= (best_candidate_df["z_count"] - 1))
).astype(float) * 0.75
best_candidate_df["outlier_score"] = (
    best_candidate_df["area_outlier_score"]
    + best_candidate_df["solidity_low_score"]
    + best_candidate_df["components_high_score"]
    + best_candidate_df["border_score"]
    + best_candidate_df["edge_z_score"]
)

likely_outliers = (
    best_candidate_df
    .sort_values(["outlier_score", "file_id"], ascending=[False, True])
    .head(OUTLIER_FILE_COUNT)
    .copy()
)

outlier_paths = []
if not likely_outliers.empty:
    fig, axes = plt.subplots(
        len(likely_outliers),
        2,
        figsize=(10.5, 3.4 * len(likely_outliers)),
        constrained_layout=True,
    )
    axes = np.asarray(axes)
    if axes.ndim == 1:
        axes = axes[None, :]

    for row_idx, row in enumerate(likely_outliers.itertuples(index=False)):
        abs_path = ROOT / str(row.file_path)
        plane = mqh.load_plane_channels(abs_path, z_index=int(row.z_index))
        dapi = plane["dapi"]
        bf = plane["brightfield"]
        mask = tifffile.imread(ROOT / str(row.mask_path)).astype(bool)

        bf_display = (
            mqh.robust_rescale(bf)
            if bf is not None
            else np.zeros_like(dapi, dtype=np.float32)
        )
        dapi_display = mqh.robust_rescale(dapi)

        reason_parts = []
        if float(row.area_outlier_score) >= 1.5:
            reason_parts.append("area outlier")
        if float(row.solidity_low_score) >= 1.5:
            reason_parts.append("low solidity")
        if int(row.raw_component_count) > 1:
            reason_parts.append(f"{int(row.raw_component_count)} raw comps")
        if bool(row.touches_border):
            reason_parts.append("touches border")
        if int(row.z_index) in {0, int(row.z_count) - 1}:
            reason_parts.append("edge z")
        if not reason_parts:
            reason_parts.append("high composite score")
        reason_text = ", ".join(reason_parts)

        for col_idx, (img, label) in enumerate([(bf_display, "BF"), (dapi_display, "DAPI")]):
            ax = axes[row_idx, col_idx]
            ax.imshow(img, cmap="gray", vmin=0.0, vmax=1.0)
            mqh.plot_mask_outline(ax, mask, color="yellow", linewidth=1.25)
            ax.set_xticks([])
            ax.set_yticks([])
            title = (
                f"file {int(row.file_id):02d} | z={int(row.z_index)} | {label}\n"
                f"score={float(row.outlier_score):.2f} | {reason_text}"
            )
            ax.set_title(title, fontsize=9)

    outlier_path = OUTLIER_QC_DIR / "likely_outlier_masks.png"
    fig.savefig(outlier_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    outlier_paths.append(outlier_path)

print(f"Wrote {len(outlier_paths)} outlier-review figure(s) to {OUTLIER_QC_DIR}")
if RENDER_INLINE_DEBUG_EXAMPLES:
    for outlier_path in outlier_paths:
        print(outlier_path.relative_to(ROOT))
        display(Image(filename=str(outlier_path)))
else:
    print("Inline outlier-review display skipped because OPTIMIZATION_MODE=True.")


## Next Step

Run `02_whole_morph_geometry.ipynb` next. That notebook reviews whole-morph geometry on every image and z plane, without choosing a final z representation yet.
